# 🍕 Phase 1: Statistics with Pizza Store Data

**Complete Statistics Course** covering:
- M1: Descriptive Statistics
- M2: Probability
- M3: Distributions
- M4: Inferential Statistics
- M5: Correlation
- M6: Regression

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import seaborn as sns

plt.style.use('seaborn-v0_8-darkgrid')
np.random.seed(42)

In [ ]:
# Create Pizza Store Dataset
n_stores = 50

df = pd.DataFrame({
    'store_id': [f'Store_{i+1}' for i in range(n_stores)],
    'city': np.random.choice(['New York', 'Chicago', 'LA', 'Houston', 'Phoenix'], n_stores),
    'daily_sales': np.round(np.random.normal(500, 120, n_stores)).astype(int),
    'avg_order_value': np.round(np.random.normal(18, 4, n_stores), 2),
    'customer_rating': np.round(np.random.uniform(2.5, 5.0, n_stores), 1),
    'delivery_time_min': np.round(np.random.exponential(25, n_stores) + 10, 1),
    'num_employees': np.random.randint(5, 25, n_stores),
})

# Add outliers for learning
df.loc[0, 'daily_sales'] = 1200  # flagship store
df.loc[1, 'daily_sales'] = 80    # struggling store

print(f'Dataset: {len(df)} pizza stores')
df.head(10)

---
## M1: Descriptive Statistics

**Goal:** Summarize data with numbers (center, spread, shape)

### 1.1 Measures of Center: Mean, Median, Mode

In [ ]:
# Mean - arithmetic average
mean_sales = df['daily_sales'].mean()
print(f"Mean daily sales: ${mean_sales:.2f}")
print(f"  Calculation: Sum / Count = {df['daily_sales'].sum()} / {len(df)} = {mean_sales:.2f}")

# Median - middle value (robust to outliers)
median_sales = df['daily_sales'].median()
print(f"\nMedian daily sales: ${median_sales:.2f}")
print(f"  (50th percentile - half stores above, half below)")

# Mode - most frequent value
mode_sales = df['daily_sales'].mode()[0]
print(f"\nMode daily sales: ${mode_sales}")

# Compare: Mean vs Median tells us about skewness
print(f"\n📊 Mean - Median = {mean_sales - median_sales:.2f}")
print("  Positive → Right-skewed (outliers pulling mean up)")

### 1.2 Measures of Spread: Variance, Std Dev, Range, IQR

In [ ]:
# Variance - average squared deviation from mean
variance = df['daily_sales'].var()
print(f"Variance: {variance:.2f}")

# Standard Deviation - square root of variance (same units as data)
std_dev = df['daily_sales'].std()
print(f"Standard Deviation: {std_dev:.2f}")
print(f"  Calculation: sqrt(Variance) = sqrt({variance:.2f}) = {std_dev:.2f}")

# Range - max minus min (sensitive to outliers)
range_sales = df['daily_sales'].max() - df['daily_sales'].min()
print(f"\nRange: {range_sales}")
print(f"  Calculation: Max - Min = {df['daily_sales'].max()} - {df['daily_sales'].min()} = {range_sales}")

# IQR - Interquartile Range (robust to outliers)
Q1 = df['daily_sales'].quantile(0.25)
Q3 = df['daily_sales'].quantile(0.75)
IQR = Q3 - Q1
print(f"\nIQR: {IQR}")
print(f"  Calculation: Q3 - Q1 = {Q3} - {Q1} = {IQR}")
print(f"  Middle 50% of stores sell between ${Q1:.0f} and ${Q3:.0f}")

### 1.3 Visualizing Distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Histogram
axes[0].hist(df['daily_sales'], bins=15, edgecolor='black', alpha=0.7)
axes[0].axvline(mean_sales, color='red', linestyle='--', label=f'Mean: {mean_sales:.0f}')
axes[0].axvline(median_sales, color='green', linestyle='--', label=f'Median: {median_sales:.0f}')
axes[0].set_title('Distribution of Daily Sales')
axes[0].set_xlabel('Daily Sales ($)')
axes[0].legend()

# Box Plot
axes[1].boxplot(df['daily_sales'], vert=True)
axes[1].set_title('Box Plot - Daily Sales')
axes[1].set_ylabel('Daily Sales ($)')

# Box plot anatomy
axes[2].boxplot(df['daily_sales'], vert=True)
axes[2].annotate('Q3 (75th)', xy=(1.1, Q3), fontsize=10)
axes[2].annotate('Median', xy=(1.1, median_sales), fontsize=10)
axes[2].annotate('Q1 (25th)', xy=(1.1, Q1), fontsize=10)
axes[2].set_title('Box Plot Anatomy')

plt.tight_layout()
plt.show()

---
## M2: Probability

**Goal:** Quantify uncertainty and likelihood of events

### 2.1 Basic Probability

In [ ]:
# Probability = Favorable outcomes / Total outcomes

# What's the probability a random store has rating >= 4.0?
high_rated = (df['customer_rating'] >= 4.0).sum()
total = len(df)
prob_high_rating = high_rated / total

print(f"P(rating >= 4.0) = {high_rated}/{total} = {prob_high_rating:.2%}")

# What's the probability a store is in New York?
ny_stores = (df['city'] == 'New York').sum()
prob_ny = ny_stores / total
print(f"P(city = New York) = {ny_stores}/{total} = {prob_ny:.2%}")

### 2.2 Conditional Probability

P(A|B) = P(A and B) / P(B)

In [ ]:
# P(high rating | New York) = P(high rating AND New York) / P(New York)

# Stores that are both high-rated AND in New York
high_and_ny = ((df['customer_rating'] >= 4.0) & (df['city'] == 'New York')).sum()

# Conditional probability
prob_high_given_ny = high_and_ny / ny_stores if ny_stores > 0 else 0

print(f"P(rating >= 4.0 | New York)")
print(f"  = P(high AND NY) / P(NY)")
print(f"  = ({high_and_ny}/{total}) / ({ny_stores}/{total})")
print(f"  = {high_and_ny}/{ny_stores}")
print(f"  = {prob_high_given_ny:.2%}")

# Compare to overall probability
print(f"\nOverall P(rating >= 4.0) = {prob_high_rating:.2%}")
print(f"P(rating >= 4.0 | New York) = {prob_high_given_ny:.2%}")

### 2.3 Bayes' Theorem

P(A|B) = P(B|A) × P(A) / P(B)

In [ ]:
# Business question: If a store has high sales, what's the probability it's in New York?

# Define "high sales" as above median
high_sales = df['daily_sales'] > median_sales

# P(NY | high sales) = P(high sales | NY) × P(NY) / P(high sales)
p_high_sales_given_ny = (high_sales & (df['city'] == 'New York')).sum() / ny_stores if ny_stores > 0 else 0
p_ny = ny_stores / total
p_high_sales = high_sales.sum() / total

p_ny_given_high_sales = (p_high_sales_given_ny * p_ny) / p_high_sales if p_high_sales > 0 else 0

print("Bayes' Theorem Application:")
print(f"P(NY | high sales) = P(high sales | NY) × P(NY) / P(high sales)")
print(f"                   = {p_high_sales_given_ny:.3f} × {p_ny:.3f} / {p_high_sales:.3f}")
print(f"                   = {p_ny_given_high_sales:.2%}")

---
## M3: Distributions

**Goal:** Understand common probability distributions

### 3.1 Normal Distribution

Bell curve - most common in nature. Defined by mean (μ) and std dev (σ).

In [ ]:
# Daily sales follow approximately normal distribution
mu = df['daily_sales'].mean()
sigma = df['daily_sales'].std()

print(f"Daily Sales ~ Normal(μ={mu:.1f}, σ={sigma:.1f})")

# 68-95-99.7 Rule
print(f"\n68-95-99.7 Rule:")
print(f"  68% of stores: ${mu-sigma:.0f} to ${mu+sigma:.0f}")
print(f"  95% of stores: ${mu-2*sigma:.0f} to ${mu+2*sigma:.0f}")
print(f"  99.7% of stores: ${mu-3*sigma:.0f} to ${mu+3*sigma:.0f}")

# Visualize
x = np.linspace(mu - 4*sigma, mu + 4*sigma, 100)
y = stats.norm.pdf(x, mu, sigma)

plt.figure(figsize=(10, 5))
plt.plot(x, y, 'b-', linewidth=2)
plt.fill_between(x, y, where=(x >= mu-sigma) & (x <= mu+sigma), alpha=0.3, label='68%')
plt.fill_between(x, y, where=(x >= mu-2*sigma) & (x <= mu+2*sigma), alpha=0.2, label='95%')
plt.axvline(mu, color='red', linestyle='--', label=f'Mean: {mu:.0f}')
plt.title('Normal Distribution of Daily Sales')
plt.xlabel('Daily Sales ($)')
plt.ylabel('Probability Density')
plt.legend()
plt.show()

### 3.2 Z-Score: Standardizing Values

Z = (X - μ) / σ

In [ ]:
# Z-score tells us how many standard deviations from the mean

# Calculate z-score for the flagship store (Store_1 with $1200 sales)
flagship_sales = 1200
z_flagship = (flagship_sales - mu) / sigma

print(f"Flagship store sales: ${flagship_sales}")
print(f"Z-score = (X - μ) / σ = ({flagship_sales} - {mu:.1f}) / {sigma:.1f} = {z_flagship:.2f}")
print(f"\nInterpretation: {z_flagship:.2f} standard deviations above the mean")

# What percentile is this?
percentile = stats.norm.cdf(z_flagship) * 100
print(f"Percentile: {percentile:.1f}% (better than {percentile:.1f}% of stores)")

# Calculate z-score for struggling store
struggling_sales = 80
z_struggling = (struggling_sales - mu) / sigma
percentile_struggling = stats.norm.cdf(z_struggling) * 100
print(f"\nStruggling store: Z = {z_struggling:.2f}, Percentile: {percentile_struggling:.1f}%")

### 3.3 Binomial Distribution

Number of successes in n independent trials, each with probability p.

In [ ]:
from scipy.special import comb
# Example: If 30% of customers order extra toppings, what's the probability
# that exactly 5 out of 10 customers order extra toppings?

n = 10  # number of trials
p = 0.30  # probability of success
k = 5  # number of successes

# P(X = k) = C(n,k) × p^k × (1-p)^(n-k)
prob = stats.binom.pmf(k, n, p)

print(f"Binomial Distribution: X ~ Binomial(n={n}, p={p})")
print(f"\nP(exactly {k} customers order extra toppings)")
print(f"  = C({n},{k}) × {p}^{k} × {1-p}^{n-k}")
print(f"  = {comb(n, k):.0f} × {p**k:.6f} × {(1-p)**(n-k):.6f}")
print(f"  = {prob:.4f} = {prob*100:.2f}%")

# Visualize
x = np.arange(0, n+1)
y = stats.binom.pmf(x, n, p)

plt.figure(figsize=(10, 5))
plt.bar(x, y, color='steelblue', edgecolor='black')
plt.axvline(n*p, color='red', linestyle='--', label=f'Expected: {n*p}')
plt.title(f'Binomial Distribution (n={n}, p={p})')
plt.xlabel('Number of Successes')
plt.ylabel('Probability')
plt.legend()
plt.show()

### 3.4 Poisson Distribution

Number of events in a fixed interval when events occur at a constant rate.

In [ ]:
# Example: A store gets an average of 8 complaints per week.
# What's the probability of getting exactly 5 complaints?

lambda_rate = 8  # average rate
k = 5  # number of events

# P(X = k) = (λ^k × e^(-λ)) / k!
prob = stats.poisson.pmf(k, lambda_rate)

print(f"Poisson Distribution: X ~ Poisson(λ={lambda_rate})")
print(f"\nP(exactly {k} complaints)")
print(f"  = (λ^k × e^(-λ)) / k!")
print(f"  = ({lambda_rate}^{k} × e^(-{lambda_rate})) / {k}!")
print(f"  = {prob:.4f} = {prob*100:.2f}%")

# Visualize
x = np.arange(0, 20)
y = stats.poisson.pmf(x, lambda_rate)

plt.figure(figsize=(10, 5))
plt.bar(x, y, color='coral', edgecolor='black')
plt.axvline(lambda_rate, color='red', linestyle='--', label=f'λ = {lambda_rate}')
plt.title(f'Poisson Distribution (λ={lambda_rate})')
plt.xlabel('Number of Events')
plt.ylabel('Probability')
plt.legend()
plt.show()

---
## M4: Inferential Statistics

**Goal:** Make conclusions about populations from samples

### 4.1 Hypothesis Testing Framework

In [ ]:
# Business Question: Is the average delivery time significantly different from 30 minutes?

# Step 1: State hypotheses
print("Step 1: State Hypotheses")
print("  H₀ (Null): μ = 30 minutes (no difference)")
print("  H₁ (Alternative): μ ≠ 30 minutes (there is a difference)")

# Step 2: Choose significance level
alpha = 0.05
print(f"\nStep 2: Significance Level α = {alpha}")

# Step 3: Calculate test statistic
sample_mean = df['delivery_time_min'].mean()
sample_std = df['delivery_time_min'].std()
n = len(df)
hypothesized_mean = 30

# t-statistic = (sample_mean - hypothesized_mean) / (sample_std / sqrt(n))
t_stat = (sample_mean - hypothesized_mean) / (sample_std / np.sqrt(n))
print(f"\nStep 3: Calculate t-statistic")
print(f"  t = (x̄ - μ₀) / (s / √n)")
print(f"  t = ({sample_mean:.2f} - {hypothesized_mean}) / ({sample_std:.2f} / √{n})")
print(f"  t = {t_stat:.3f}")

# Step 4: Find p-value
p_value = 2 * (1 - stats.t.cdf(abs(t_stat), df=n-1))  # two-tailed
print(f"\nStep 4: p-value = {p_value:.4f}")

# Step 5: Make decision
print(f"\nStep 5: Decision")
if p_value < alpha:
    print(f"  p-value ({p_value:.4f}) < α ({alpha}) → Reject H₀")
    print(f"  Conclusion: Delivery time IS significantly different from 30 minutes")
else:
    print(f"  p-value ({p_value:.4f}) >= α ({alpha}) → Fail to reject H₀")
    print(f"  Conclusion: No significant evidence that delivery time differs from 30 minutes")

### 4.2 T-Test (Comparing Means)

In [ ]:
# Compare delivery times between two cities
ny_delivery = df[df['city'] == 'New York']['delivery_time_min']
chicago_delivery = df[df['city'] == 'Chicago']['delivery_time_min']

print(f"New York: n={len(ny_delivery)}, mean={ny_delivery.mean():.2f} min")
print(f"Chicago: n={len(chicago_delivery)}, mean={chicago_delivery.mean():.2f} min")

# Two-sample t-test
t_stat, p_value = stats.ttest_ind(ny_delivery, chicago_delivery)

print(f"\nTwo-Sample T-Test:")
print(f"  t-statistic = {t_stat:.3f}")
print(f"  p-value = {p_value:.4f}")

if p_value < 0.05:
    print("  → Significant difference in delivery times between cities")
else:
    print("  → No significant difference in delivery times between cities")

### 4.3 Central Limit Theorem (CLT)

**The most important theorem in statistics!**

The CLT states: When you take many samples from ANY population and calculate their means, those sample means will be normally distributed, regardless of the original population's shape.

**Key implications:**
- Sample mean (x̄) → Normal distribution as n increases
- Mean of sample means = Population mean (μ)
- Standard error = σ/√n (gets smaller as n increases)

In [ ]:
# Central Limit Theorem Demonstration
# Even if original data is NOT normal, sample means WILL BE normal!

np.random.seed(42)

# Create a clearly NON-normal population (exponential - skewed right)
population = np.random.exponential(scale=20, size=10000)  # Skewed delivery times

# Take many samples and calculate their means
sample_sizes = [5, 30, 100]
n_samples = 1000

fig, axes = plt.subplots(2, 3, figsize=(14, 8))

# Top row: Show sample distributions
# Bottom row: Show distribution of sample MEANS

for idx, n in enumerate(sample_sizes):
    # Take n_samples samples of size n, calculate mean of each
    sample_means = [np.random.choice(population, size=n).mean() for _ in range(n_samples)]
    
    # Top: One sample distribution
    sample = np.random.choice(population, size=n)
    axes[0, idx].hist(sample, bins=20, color='steelblue', edgecolor='black', alpha=0.7)
    axes[0, idx].set_title(f'One Sample (n={n})')
    axes[0, idx].axvline(sample.mean(), color='red', linestyle='--', label=f'Mean={sample.mean():.1f}')
    axes[0, idx].legend()
    
    # Bottom: Distribution of sample means
    axes[1, idx].hist(sample_means, bins=30, color='coral', edgecolor='black', alpha=0.7)
    axes[1, idx].set_title(f'Distribution of {n_samples} Sample Means (n={n})')
    axes[1, idx].axvline(np.mean(sample_means), color='red', linestyle='--', 
                         label=f'Mean={np.mean(sample_means):.1f}')
    axes[1, idx].axvline(population.mean(), color='green', linestyle=':', 
                         label=f'Pop μ={population.mean():.1f}')
    axes[1, idx].legend()

plt.suptitle('Central Limit Theorem: Sample Means Become Normal!', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('=' * 60)
print('CENTRAL LIMIT THEOREM SUMMARY')
print('=' * 60)
print(f'\nPopulation: Exponential (skewed right)')
print(f'Population mean (μ): {population.mean():.2f}')
print(f'Population std (σ): {population.std():.2f}')
print(f'\nAs sample size increases:')
for n in sample_sizes:
    sample_means = [np.random.choice(population, size=n).mean() for _ in range(n_samples)]
    theoretical_se = population.std() / np.sqrt(n)
    actual_se = np.std(sample_means)
    print(f'  n={n:3d}: SE (theoretical) = σ/√n = {theoretical_se:.2f}, SE (actual) = {actual_se:.2f}')
print(f'\n✓ Sample means are normally distributed (even though population is skewed!)')
print(f'✓ This is WHY confidence intervals and t-tests work!')

### 4.4 Confidence Intervals

In [ ]:
# 95% Confidence Interval for mean daily sales
confidence = 0.95
sample_mean = df['daily_sales'].mean()
sample_std = df['daily_sales'].std()
n = len(df)

# Standard error
se = sample_std / np.sqrt(n)

# t-critical value for 95% CI
t_critical = stats.t.ppf((1 + confidence) / 2, df=n-1)

# Margin of error
margin = t_critical * se

# Confidence interval
ci_lower = sample_mean - margin
ci_upper = sample_mean + margin

print(f"95% Confidence Interval for Mean Daily Sales:")
print(f"\n  Sample mean (x̄) = ${sample_mean:.2f}")
print(f"  Standard error (SE) = s/√n = {sample_std:.2f}/√{n} = ${se:.2f}")
print(f"  t-critical (α=0.05, df={n-1}) = {t_critical:.3f}")
print(f"  Margin of error = t × SE = {t_critical:.3f} × {se:.2f} = ${margin:.2f}")
print(f"\n  95% CI: [${ci_lower:.2f}, ${ci_upper:.2f}]")
print(f"\n  Interpretation: We are 95% confident the true population mean")
print(f"  daily sales is between ${ci_lower:.2f} and ${ci_upper:.2f}")

---
## M5: Correlation

**Goal:** Measure relationships between variables

### 5.1 Pearson Correlation Coefficient

In [ ]:
# Correlation between number of employees and daily sales
x = df['num_employees']
y = df['daily_sales']

# Calculate Pearson correlation
r, p_value = stats.pearsonr(x, y)

print("Pearson Correlation: r = Σ[(xi - x̄)(yi - ȳ)] / [(n-1) × sx × sy]")
print(f"\nCorrelation between employees and sales:")
print(f"  r = {r:.3f}")
print(f"  p-value = {p_value:.4f}")

# Interpretation
print(f"\nInterpretation:")
if abs(r) < 0.3:
    strength = "weak"
elif abs(r) < 0.7:
    strength = "moderate"
else:
    strength = "strong"
direction = "positive" if r > 0 else "negative"
print(f"  {strength.capitalize()} {direction} correlation")

# Coefficient of determination
r_squared = r ** 2
print(f"\nR² = {r_squared:.3f}")
print(f"  {r_squared*100:.1f}% of variance in sales is explained by number of employees")

### 5.2 Correlation Matrix & Heatmap

In [ ]:
# Correlation matrix for all numeric variables
numeric_cols = ['daily_sales', 'avg_order_value', 'customer_rating', 'delivery_time_min', 'num_employees']
corr_matrix = df[numeric_cols].corr()

print("Correlation Matrix:")
print(corr_matrix.round(3))

# Heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='RdBu_r', center=0, 
            square=True, linewidths=0.5, fmt='.2f')
plt.title('Correlation Heatmap - Pizza Store Metrics')
plt.tight_layout()
plt.show()

### 5.3 Scatter Plot with Correlation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Employees vs Sales
axes[0].scatter(df['num_employees'], df['daily_sales'], alpha=0.6)
z = np.polyfit(df['num_employees'], df['daily_sales'], 1)
p = np.poly1d(z)
axes[0].plot(df['num_employees'].sort_values(), p(df['num_employees'].sort_values()), 
             'r--', label=f'r = {r:.3f}')
axes[0].set_xlabel('Number of Employees')
axes[0].set_ylabel('Daily Sales ($)')
axes[0].set_title('Employees vs Daily Sales')
axes[0].legend()

# Plot 2: Rating vs Delivery Time
r2, _ = stats.pearsonr(df['customer_rating'], df['delivery_time_min'])
axes[1].scatter(df['delivery_time_min'], df['customer_rating'], alpha=0.6, color='green')
z2 = np.polyfit(df['delivery_time_min'], df['customer_rating'], 1)
p2 = np.poly1d(z2)
axes[1].plot(df['delivery_time_min'].sort_values(), p2(df['delivery_time_min'].sort_values()), 
             'r--', label=f'r = {r2:.3f}')
axes[1].set_xlabel('Delivery Time (min)')
axes[1].set_ylabel('Customer Rating')
axes[1].set_title('Delivery Time vs Rating')
axes[1].legend()

plt.tight_layout()
plt.show()

---
## M6: Regression

**Goal:** Predict one variable from another

### 6.1 Simple Linear Regression

y = β₀ + β₁x + ε

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

# Predict daily sales from number of employees
X = df[['num_employees']]
y = df['daily_sales']

# Fit model
model = LinearRegression()
model.fit(X, y)

# Coefficients
beta_0 = model.intercept_
beta_1 = model.coef_[0]

print("Simple Linear Regression: y = β₀ + β₁x")
print(f"\nEquation: Daily Sales = {beta_0:.2f} + {beta_1:.2f} × Employees")
print(f"\nInterpretation:")
print(f"  β₀ (intercept) = {beta_0:.2f}: Base sales with 0 employees")
print(f"  β₁ (slope) = {beta_1:.2f}: Each additional employee adds ${beta_1:.2f} to daily sales")

# Predictions
y_pred = model.predict(X)

# Model evaluation
r2 = r2_score(y, y_pred)
rmse = np.sqrt(mean_squared_error(y, y_pred))

print(f"\nModel Performance:")
print(f"  R² = {r2:.3f} ({r2*100:.1f}% of variance explained)")
print(f"  RMSE = ${rmse:.2f} (average prediction error)")

### 6.2 Regression Visualization

In [ ]:
plt.figure(figsize=(10, 6))

# Scatter plot
plt.scatter(df['num_employees'], df['daily_sales'], alpha=0.6, label='Actual')

# Regression line
x_line = np.linspace(df['num_employees'].min(), df['num_employees'].max(), 100)
y_line = beta_0 + beta_1 * x_line
plt.plot(x_line, y_line, 'r-', linewidth=2, label=f'y = {beta_0:.1f} + {beta_1:.1f}x')

# Confidence band (simplified)
plt.fill_between(x_line, y_line - rmse, y_line + rmse, alpha=0.2, color='red', label='±1 RMSE')

plt.xlabel('Number of Employees')
plt.ylabel('Daily Sales ($)')
plt.title(f'Linear Regression: Sales vs Employees (R² = {r2:.3f})')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### 6.3 Multiple Linear Regression

In [ ]:
# Predict sales from multiple features
X_multi = df[['num_employees', 'avg_order_value', 'customer_rating']]
y = df['daily_sales']

model_multi = LinearRegression()
model_multi.fit(X_multi, y)

print("Multiple Linear Regression:")
print(f"\nDaily Sales = {model_multi.intercept_:.2f}", end='')
for feat, coef in zip(X_multi.columns, model_multi.coef_):
    sign = '+' if coef >= 0 else ''
    print(f" {sign} {coef:.2f}×{feat}", end='')
print()

# Predictions and evaluation
y_pred_multi = model_multi.predict(X_multi)
r2_multi = r2_score(y, y_pred_multi)
rmse_multi = np.sqrt(mean_squared_error(y, y_pred_multi))

print(f"\nModel Performance:")
print(f"  R² = {r2_multi:.3f} (improved from {r2:.3f})")
print(f"  RMSE = ${rmse_multi:.2f} (improved from ${rmse:.2f})")

# Feature importance
print(f"\nFeature Importance (by coefficient magnitude):")
for feat, coef in sorted(zip(X_multi.columns, model_multi.coef_), key=lambda x: abs(x[1]), reverse=True):
    print(f"  {feat}: {coef:.2f}")

### 6.4 Residual Analysis

In [ ]:
# Residuals = Actual - Predicted
residuals = y - y_pred

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Residuals vs Fitted
axes[0].scatter(y_pred, residuals, alpha=0.6)
axes[0].axhline(y=0, color='r', linestyle='--')
axes[0].set_xlabel('Fitted Values')
axes[0].set_ylabel('Residuals')
axes[0].set_title('Residuals vs Fitted')

# Histogram of residuals
axes[1].hist(residuals, bins=15, edgecolor='black', alpha=0.7)
axes[1].axvline(x=0, color='r', linestyle='--')
axes[1].set_xlabel('Residuals')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Distribution of Residuals')

# Q-Q plot
stats.probplot(residuals, dist="norm", plot=axes[2])
axes[2].set_title('Q-Q Plot (Normality Check)')

plt.tight_layout()
plt.show()

print("Residual Analysis:")
print(f"  Mean of residuals: {residuals.mean():.4f} (should be ~0)")
print(f"  Std of residuals: {residuals.std():.2f}")

---
## Summary

You've learned:
- **Descriptive Stats**: Mean, median, std dev, IQR
- **Probability**: Basic, conditional, Bayes
- **Distributions**: Normal, Binomial, Poisson
- **Inference**: Hypothesis testing, t-tests, confidence intervals
- **Correlation**: Pearson r, heatmaps
- **Regression**: Simple and multiple linear regression